### 1. Write a program to make a statistical estimate for π using one of the two methods suggested in ‘Maths13’

I will estimate $\pi$ using a Monte-Carlo simualtion of a $[1,1]$ square.

In [ ]:
import numpy as np

seed = np.random.default_rng(1234)

def estimate_pi(N, r=1.0):
    x = seed.random(N)
    y = seed.random(N)
    inside = np.count_nonzero(x**2 + y**2 < r**2)
    return 4 / r**2 * inside / N


for N in [10**2, 10**4, 10**6, 10**8]:
    pi_hat = estimate_pi(N)
    print(f"N = {N}: pi_hat = {pi_hat:.5f}, error = {pi_hat - np.pi:+.5f}")

N = 100: pi_hat = 3.28000, error = +0.13841
N = 10000: pi_hat = 3.15640, error = +0.01481
N = 1000000: pi_hat = 3.13995, error = -0.00164
N = 100000000: pi_hat = 3.14145, error = -0.00014


### 2. How accurate is the Monte Carlo?

Repeat pi estimate many times, fit the spread, and check 1.64/sqrt(N)

In [44]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm

plot_dir = "out/plots"

repeats = 2000


# Compare the estimates with a Gaussian distribution.
N = 10_000
estimates = []

for _ in range(repeats):
    estimates.append(estimate_pi(N))

estimates = np.array(estimates)
mean, sigma = norm.fit(estimates)

p = np.pi / 4
theory_sigma = 4 * np.sqrt(p * (1 - p) / N)

print(f"N = {N}")
print(f"Fitted mean: {mean:.4f}")
print(f"Fitted standard deviation: {sigma:.4f}")
print(f"Expected standard deviation: {theory_sigma:.4f}")

# Estimates are spaced by 4/N. Put eight possible values in each bin.
step = 4 / N
smallest_count = round(estimates.min() / step)
largest_count = round(estimates.max() / step)

values_per_bin = 8
number_of_bins = (largest_count - smallest_count) // values_per_bin + 1

edges = step * (
    smallest_count - 0.5
    + values_per_bin * np.arange(number_of_bins + 1)
)

x_values = np.linspace(estimates.min(), estimates.max(), 300)

plt.figure()
plt.hist(
    estimates,
    bins=edges,
    density=True,
    alpha=0.5,
    label=f"{repeats} runs",
)
plt.plot(x_values, norm.pdf(x_values, mean, sigma), label="Gaussian fit")
plt.axvline(np.pi, color="black", linestyle="--", label=r"$\pi$")
plt.xlabel(r"$\hat{\pi}$")
plt.ylabel("Density")
plt.legend()
plt.tight_layout()
plt.savefig(plot_dir + "/pi_histogram.png", dpi=150)
plt.close()


# Check how the spread changes as the sample size increases.
sample_sizes = np.array([100, 300, 1000, 3000, 10_000, 30_000])
sigmas = []

for N in sample_sizes:
    estimates = []

    for _ in range(repeats):
        estimates.append(estimate_pi(N))

    sigmas.append(np.std(estimates))

sigmas = np.array(sigmas)
theory_sigmas = 4 * np.sqrt(p * (1 - p) / sample_sizes)

slope, intercept = np.polyfit(np.log(sample_sizes), np.log(sigmas), 1)

print(f"\nMeasured slope: {slope:.3f}")
print("Expected slope: -0.5")

plt.figure()
plt.loglog(sample_sizes, sigmas, "o", label="Measured spread")
plt.loglog(
    sample_sizes,
    theory_sigmas,
    label=r"$4\sqrt{p(1-p)/N}$",
)
plt.xlabel("$N$")
plt.ylabel(r"$\sigma_{\hat{\pi}}$")
plt.legend()
plt.tight_layout()
plt.savefig(plot_dir + "/pi_scaling.png", dpi=150)
plt.close()


# Compare the relative error for two circle radii.
N = 10_000

for radius in [1.0, np.sqrt(2 / 3)]:
    estimates = []

    for _ in range(repeats):
        estimates.append(estimate_pi(N, radius))

    relative_error = np.std(estimates) / np.pi
    p_inside = np.pi * radius**2 / 4

    measured = relative_error * np.sqrt(N)
    expected = np.sqrt((1 - p_inside) / p_inside)

    print(f"\nRadius squared: {radius**2:.3f}")
    print(f"Relative error . sqrt(N): {measured:.3f}")
    print(f"Expected value: {expected:.3f}")

N = 10000
Fitted mean: 3.1413
Fitted standard deviation: 0.0165
Expected standard deviation: 0.0164

Measured slope: -0.500
Expected slope: -0.5

Radius squared: 1.000
Relative error . sqrt(N): 0.510
Expected value: 0.523

Radius squared: 0.667
Relative error . sqrt(N): 0.923
Expected value: 0.954
